<a href="https://colab.research.google.com/github/AmnonElias/DeepLearningProject/blob/main/motion_gating.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import cv2
import os
from google.colab import drive
drive.mount('/content/drive')

def run_adaptive_motion_detection(video_source, camera_type='RGB', output_name='adaptive_output.avi', target_fps=2):
    # 1. הגדרת פרמטרים לפי סוג המצלמה
    if camera_type == 'IR':
        print("מזהה מצלמת IR: מפעיל רגישות גבוהה וסינון רעשים...")
        VAR_THRESH = 16
        MIN_AREA = 20
        USE_MORPHOLOGY = True
    elif camera_type == 'RGB':
        print("מזהה מצלמת RGB: מפעיל רגישות סטנדרטית למניעת התראות שווא...")
        VAR_THRESH = 50
        MIN_AREA = 500
        USE_MORPHOLOGY = False
    else:
        print("שגיאה: סוג מצלמה לא מוכר.")
        return

    cap = cv2.VideoCapture(video_source)
    if not cap.isOpened(): return

    original_fps = cap.get(cv2.CAP_PROP_FPS)
    if original_fps == 0: original_fps = 30
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*'DIVX')
    out = cv2.VideoWriter(output_name, fourcc, target_fps, (width, height))
    frame_skip_interval = int(original_fps / target_fps)

    # אתחול מחסר הרקע עם הפרמטרים המותאמים
    backSub = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=VAR_THRESH, detectShadows=False)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret: break

        frame_count += 1
        if frame_count % frame_skip_interval != 0: continue

        # אם זו מצלמת IR, נפעיל את העיבוד המקדים (טשטוש)
        if camera_type == 'IR':
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            processed_frame = cv2.GaussianBlur(gray, (5, 5), 0)
        else:
            # ב-RGB אין צורך בטשטוש, שולחים את הפריים כפי שהוא
            processed_frame = frame

        # הפעלת זיהוי התנועה
        fgMask = backSub.apply(processed_frame)
        _, fgMask = cv2.threshold(fgMask, 200, 255, cv2.THRESH_BINARY)

        # הרחבת פיקסלים (רק ב-IR)
        if USE_MORPHOLOGY:
            fgMask = cv2.dilate(fgMask, kernel, iterations=2)

        contours, _ = cv2.findContours(fgMask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        motion_detected = False

        for contour in contours:
            # סינון לפי שטח מינימלי (מותאם לסוג המצלמה)
            if cv2.contourArea(contour) < MIN_AREA:
                continue

            motion_detected = True
            x, y, w, h = cv2.boundingRect(contour)
            # צבע הריבוע ישתנה לפי סוג המצלמה לנוחות הבדיקה
            box_color = (0, 255, 0) if camera_type == 'RGB' else (255, 0, 0) # ירוק ל-RGB, כחול ל-IR
            cv2.rectangle(frame, (x, y), (x+w, y+h), box_color, 2)

        if motion_detected:
            cv2.putText(frame, f"MOTION! ({camera_type})", (10, 50),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)

        out.write(frame)

    cap.release()
    out.release()
    print(f"העיבוד הסתיים! ({camera_type})")

# הרצה:

run_adaptive_motion_detection('/content/drive/My Drive/DeepLearning/Videos/davinci_a_cat_running_outside_a_house__simulating__a_cctv_.mp4', camera_type='IR', output_name='/content/drive/My Drive/DeepLearning/Videos/motion_result_davinci_a_cat_running_outside_a_house__simulating__a_cctv.avi', target_fps=4)
run_adaptive_motion_detection('/content/drive/My Drive/DeepLearning/Videos/0_Mouse_Mice_1280x720.mp4', camera_type='RGB', output_name='/content/drive/My Drive/DeepLearning/Videos/motion_result_0_Mouse_Mice_1280x720.avi', target_fps=2)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
מזהה מצלמת IR: מפעיל רגישות גבוהה וסינון רעשים...
העיבוד הסתיים! (IR)
מזהה מצלמת RGB: מפעיל רגישות סטנדרטית למניעת התראות שווא...
העיבוד הסתיים! (RGB)
